# PML · Lecture 8 — Basis functions, the design matrix, and Bayesian linear regression

**Companion notebook.** In the lecture you built the design matrix $\Phi$ *by hand*. Here you'll do the whole pipeline in code and *see* it: take a raw dataset, turn it into a **design matrix** with **basis functions**, and use a matrix library (NumPy) to **solve** for the weights — first plain least squares, then ridge (= a Gaussian prior), then the full **Bayesian** posterior with honest error bars that widen where there is no data. Finally you'll watch a model **update online**, its posterior becoming the prior for the next batch.

No GPU needed — this is pure `numpy` + `matplotlib` (already in Colab). Just run each cell top to bottom.

**The one idea:** the math only cares that $f(x)=w^\top\phi(x)$ is *linear in $w$*. Swap the raw input $x$ for features $\phi(x)$ and the same machinery fits curves.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)          # reproducible randomness
plt.rcParams['figure.figsize'] = (7, 4)
print('numpy', np.__version__)

## 1. From a *raw dataset* to a *design matrix*

A dataset is just inputs and targets. To fit a model we choose **features** — the columns of the **design matrix** $\Phi$. Each row is one data point; each column is one basis function evaluated at that point: $\;\Phi_{ij} = \phi_j(x_i)$.

The simplest choice is a **bias column of 1s** plus the raw input — that fits a straight line $y = w_0 + w_1 x$.

In [ ]:
# a tiny 'normal' dataset: one input x, one target y
raw = np.array([[0.0, 1.2],
                [1.0, 2.1],
                [2.0, 2.0],
                [3.0, 3.4]])
x_raw, y_raw = raw[:, 0], raw[:, 1]

# build the design matrix: column of 1s (bias)  +  the raw x
Phi_line = np.column_stack([np.ones_like(x_raw), x_raw])
print('design matrix  [1 | x]:')
print(Phi_line)
print('shape:', Phi_line.shape, ' (N rows = points, M cols = features)')

Now **solve** for the weights. The least-squares solution satisfies the *normal equations* $\;(\Phi^\top\Phi)\,w = \Phi^\top y$. We hand that to a matrix library rather than doing algebra by hand:

In [ ]:
w = np.linalg.solve(Phi_line.T @ Phi_line, Phi_line.T @ y_raw)
print('weights [w0 (intercept), w1 (slope)] =', np.round(w, 3))

That's the whole trick: **choose features → stack them into $\Phi$ → solve.** Everything below just chooses *richer* features so the same linear solver can fit curves.

## 2. A dataset that bends

We'll pretend the world follows $g(x)=\sin(2\pi x)$ but we only see noisy samples. (The model never sees $g$ — it's the dashed line for us to judge the fit.)

In [ ]:
def g(x):
    return np.sin(2*np.pi*x)

N = 12
x = np.sort(rng.uniform(0, 1, size=N))
noise_sd = 0.15
y = g(x) + rng.normal(0, noise_sd, size=N)

xs = np.linspace(0, 1, 200)             # dense grid for drawing smooth curves
plt.scatter(x, y, c='k', zorder=3, label='data')
plt.plot(xs, g(xs), 'g--', label=r'true $g(x)=\sin(2\pi x)$')
plt.legend(); plt.xlabel('x'); plt.ylabel('y'); plt.title('A dataset that bends'); plt.show()

## 3. Polynomial basis → design matrix

Polynomial features are $\phi_j(x) = x^j$, giving columns $1, x, x^2, \dots, x^d$. `np.vander` builds exactly this matrix.

In [ ]:
def poly_design(x, degree):
    """N x (degree+1) matrix with columns 1, x, x^2, ..., x^degree."""
    return np.vander(np.asarray(x, float), degree + 1, increasing=True)

Phi = poly_design(x, degree=3)
print('Phi shape:', Phi.shape)
print('first 3 rows (1, x, x^2, x^3):')
print(np.round(Phi[:3], 3))

## 4. Solve with a matrix library — three equivalent views

- **Normal equations:** `np.linalg.solve(Phi.T@Phi, Phi.T@y)`
- **Least squares directly:** `np.linalg.lstsq(Phi, y)` — *preferred*, it never forms $\Phi^\top\Phi$ (which squares the conditioning) and is more numerically stable.
- **Textbook formula:** $w=(\Phi^\top\Phi)^{-1}\Phi^\top y$ with `np.linalg.inv` — correct, but avoid `inv` in real code.

In [ ]:
def fit_lstsq(Phi, y):
    w, *_ = np.linalg.lstsq(Phi, y, rcond=None)
    return w

w_solve = np.linalg.solve(Phi.T @ Phi, Phi.T @ y)     # normal equations
w_lstsq = fit_lstsq(Phi, y)                           # least squares
print('normal-equations w:', np.round(w_solve, 3))
print('lstsq w           :', np.round(w_lstsq, 3))
print('max abs difference:', np.max(np.abs(w_solve - w_lstsq)))

yhat = poly_design(xs, 3) @ w_lstsq                   # predict on the grid: y = Phi w
plt.scatter(x, y, c='k', zorder=3, label='data')
plt.plot(xs, g(xs), 'g--', label='true')
plt.plot(xs, yhat, 'b', label='degree-3 fit')
plt.legend(); plt.title('Least-squares polynomial fit'); plt.show()

## 5. Flexibility and overfitting

More columns = more flexibility. A high-degree polynomial can pass through every point but wiggles wildly between them — it's fitting the **noise**.

In [ ]:
for d in [1, 3, 9]:
    w = fit_lstsq(poly_design(x, d), y)
    plt.plot(xs, poly_design(xs, d) @ w, label=f'degree {d}')
plt.scatter(x, y, c='k', zorder=3)
plt.plot(xs, g(xs), 'g--', lw=1)
plt.ylim(-2, 2); plt.legend(); plt.title('Higher degree -> more wiggle (overfitting)'); plt.show()

## 6. Gaussian RBF basis (the better default)

A radial basis function is a localized bump, $\phi_j(x)=\exp\!\big(-\tfrac{(x-\mu_j)^2}{2s^2}\big)$, centered at $\mu_j$ with width $s$. We build the whole design matrix at once with **broadcasting** — no Python loop over points.

In [ ]:
def rbf_design(x, centers, s, bias=True):
    x = np.asarray(x, float)[:, None]          # (N, 1)
    c = np.asarray(centers, float)[None, :]    # (1, M)
    Phi = np.exp(-(x - c)**2 / (2*s**2))        # (N, M) — one bump per column
    if bias:
        Phi = np.column_stack([np.ones(x.shape[0]), Phi])
    return Phi

centers = np.linspace(0, 1, 8)
s = 0.12
for mu in centers:
    plt.plot(xs, np.exp(-(xs-mu)**2/(2*s**2)), lw=1)
plt.title('8 Gaussian RBF bumps across the input range'); plt.xlabel('x'); plt.show()

print('RBF design matrix shape (with bias):', rbf_design(x, centers, s).shape)

## 7. Ridge = a Gaussian prior

With a flexible basis we need to discourage large weights. **Ridge** adds a penalty $\lambda\lVert w\rVert^2$, solved by $w=(\Phi^\top\Phi+\lambda I)^{-1}\Phi^\top y$. From the lecture, this penalty *is* a zero-mean Gaussian prior $w\sim\mathcal N(0,\alpha^{-1}I)$, with $\;\lambda=\alpha/\beta=\alpha\sigma^2$.

In [ ]:
def ridge_fit(Phi, y, lam):
    M = Phi.shape[1]
    return np.linalg.solve(Phi.T @ Phi + lam*np.eye(M), Phi.T @ y)

Phi_tr = rbf_design(x, centers, s)
for lam in [0.0, 1e-3, 1.0]:
    w = ridge_fit(Phi_tr, y, lam)
    plt.plot(xs, rbf_design(xs, centers, s) @ w, label=f'lambda={lam}')
plt.scatter(x, y, c='k', zorder=3); plt.plot(xs, g(xs), 'g--', lw=1)
plt.ylim(-1.8, 1.8); plt.legend()
plt.title('RBF fit: ridge lambda (= alpha/beta) tames the wiggle'); plt.show()

## 8. Full Bayesian linear regression — a *distribution* over fits

Instead of one weight vector, keep the whole Gaussian posterior. With prior precision $\alpha$ and noise precision $\beta=1/\sigma^2$:

$$S_N=(\alpha I+\beta\,\Phi^\top\Phi)^{-1},\qquad m_N=\beta\,S_N\,\Phi^\top y.$$

The **predictive** distribution at a new $x_\*$ has mean $m_N^\top\phi(x_\*)$ and variance $\;\sigma_N^2(x_\*)=\tfrac1\beta+\phi(x_\*)^\top S_N\,\phi(x_\*)$ — a fixed noise floor plus a term that **grows away from the data**.

In [ ]:
def blr_posterior(Phi, y, alpha, beta):
    M = Phi.shape[1]
    S_N = np.linalg.inv(alpha*np.eye(M) + beta * Phi.T @ Phi)
    m_N = beta * S_N @ Phi.T @ y
    return m_N, S_N

def predictive(Phi_star, m_N, S_N, beta):
    mean = Phi_star @ m_N
    var  = 1.0/beta + np.sum((Phi_star @ S_N) * Phi_star, axis=1)   # phi^T S_N phi per row
    return mean, np.sqrt(var)

alpha, beta = 1.0, 1.0/noise_sd**2
m_N, S_N = blr_posterior(Phi_tr, y, alpha, beta)

# 'ridge = prior': ridge with lambda = alpha/beta reproduces the posterior MEAN exactly
w_ridge = ridge_fit(Phi_tr, y, alpha/beta)
print('ridge(lambda=alpha/beta) vs posterior mean m_N  ->  max abs diff =',
      np.max(np.abs(w_ridge - m_N)))

xg = np.linspace(-0.3, 1.3, 300)             # extend PAST the data to see the band fan out
mean, sd = predictive(rbf_design(xg, centers, s), m_N, S_N, beta)
plt.axvspan(0, 1, color='k', alpha=0.04, label='data region')
plt.plot(xg, g(xg), 'g--', label='true')
plt.plot(xg, mean, 'b', label='predictive mean')
plt.fill_between(xg, mean-2*sd, mean+2*sd, color='b', alpha=0.15, label=r'$\pm 2\sigma$')
plt.scatter(x, y, c='k', zorder=3, label='data')
plt.ylim(-2.5, 2.5); plt.legend()
plt.title('Bayesian LR: error bars widen away from the data'); plt.show()

The printed difference is ~0: **ridge with $\lambda=\alpha/\beta$ is exactly the posterior mean** — the lecture's 'regularization *is* a Gaussian prior', shown numerically. And the band is **tight where points sit** and **flares past the edges** (we plotted out to $x\in[-0.3,1.3]$) — the model is honest about what it doesn't know. Try lowering `N` or deleting a chunk of the data and re-running: the band widens there.

> **Choosing $\alpha,\beta$.** Here we set them by hand. In practice you can pick them by maximizing the **evidence** (marginal likelihood) $p(y\mid\alpha,\beta)$ — *empirical Bayes*, covered in the lecture.

## 9. Online / sequential updating — *today's posterior is tomorrow's prior*

Split the data into two batches. Fit batch 1 from the $\alpha$-prior to get $(m_1,S_1)$. Then fit batch 2 **using $(m_1,S_1)$ as the prior**:

$$S=(S_1^{-1}+\beta\,\Phi^\top\Phi)^{-1},\qquad m=S\,(S_1^{-1}m_1+\beta\,\Phi^\top y).$$

No need to re-read batch 1 — and the result is **identical** to fitting all the data at once.

In [ ]:
order = np.argsort(x)
b1, b2 = order[:N//2], order[N//2:]         # two batches

m1, S1 = blr_posterior(rbf_design(x[b1], centers, s), y[b1], alpha, beta)   # batch 1 from alpha-prior

Phi2   = rbf_design(x[b2], centers, s)      # batch 2, prior = (m1, S1)
S1_inv = np.linalg.inv(S1)
S2 = np.linalg.inv(S1_inv + beta * Phi2.T @ Phi2)
m2 = S2 @ (S1_inv @ m1 + beta * Phi2.T @ y[b2])

for m, S, lbl, col in [(m1, S1, 'after batch 1', 'orange'), (m2, S2, 'after batch 2', 'b')]:
    mean, sd = predictive(rbf_design(xg, centers, s), m, S, beta)
    plt.plot(xg, mean, col, label=lbl)
    plt.fill_between(xg, mean-2*sd, mean+2*sd, color=col, alpha=0.12)
plt.scatter(x[b1], y[b1], c='orange', zorder=3, label='batch 1')
plt.scatter(x[b2], y[b2], c='b', zorder=3, label='batch 2')
plt.plot(xg, g(xg), 'g--', lw=1); plt.ylim(-2.5, 2.5); plt.legend()
plt.title('Online update: posterior after batch 1 -> prior for batch 2'); plt.show()

# sequential == all-at-once (since batch 1 started from the same alpha-prior)
m_all, S_all = blr_posterior(rbf_design(x, centers, s), y, alpha, beta)
print('max |m_sequential - m_all_at_once| =', np.max(np.abs(m2 - m_all)))

After batch 1 the band is wide (little data); after batch 2 it tightens where the new points landed — the posterior only ever gets *more* certain. And the printed difference is ~0: processing the batches one-at-a-time equals processing them together.

## 10. Your turn

1. **Design matrix by hand vs code.** Rebuild the lecture's example: inputs $x=(0,1,2)$, two RBFs at $\mu=0,2$ with $s=1$ (set `bias=False`). Print `rbf_design([0,1,2], [0,2], 1.0, bias=False)` and check it matches $\begin{bmatrix}1 & e^{-2}\\ e^{-1/2}&e^{-1/2}\\ e^{-2}&1\end{bmatrix}$.
2. **Prior strength.** Sweep `alpha` in `blr_posterior` over e.g. `[0.01, 1, 100]` and plot the predictive mean+band for each. What does a strong prior (large $\alpha$) do to the fit and the band?
3. **Fewer points / a gap.** Remove all data with `0.4 < x < 0.7` and refit the Bayesian model. Where does the uncertainty band get wide, and why?
4. **Width $s$.** Change the RBF width `s` (try `0.05` and `0.3`). How does it trade wiggliness against smoothness?

## Recap
- A **basis function** is a fixed feature; stacking features for every point makes the **design matrix** $\Phi$.
- Fitting is a matrix solve: least squares, or **ridge** $=$ a Gaussian **prior** ($\lambda=\alpha/\beta$).
- **Bayesian** LR keeps the whole posterior, giving predictive **error bars that widen away from data**.
- **Online** learning is just: *posterior becomes the next prior*.